# Hearing-Test Funnel: Gold-Layer Analysis

This notebook reads exclusively from the **gold** layer of the bronze/silver/gold
medallion built by `sql/medallion.sql` (module M3c) — the same SQL DuckDB runs
locally and `scripts/load_to_databricks.py` runs against this workspace, so the
numbers below match the local demo exactly.

**Consent & censoring caveats (read before interpreting anything below):**
- Every bridge-linked (cross-device) number here only covers **consented,
  signed-in users** (`silver.v_attribution_eligible`) — treat channel/market
  splits on downloads, pairing and D30 as a **lower bound and a mix signal**,
  never as absolute volume for the whole user base.
- **D30 retention** excludes devices whose first app open is within 34 days of
  the dataset's last recorded event (`silver.app_user_stages.censored`) — those
  users have not had time to reach day 30 yet, so including them would bias the
  rate downward.
- Set the `catalog` variable in the next cell to wherever
  `scripts/load_to_databricks.py` loaded the data (default: `workspace`,
  matching `agent.db.DEFAULT_DATABRICKS_CATALOG`).

In [ ]:
# Point this notebook at the catalog scripts/load_to_databricks.py loaded into.
catalog = "workspace"  # change if your DATABRICKS_CATALOG differs
spark.sql(f"USE CATALOG {catalog}")

## 1. Funnel overview

Distinct users/devices at each of the 5 funnel stages, from `gold.funnel_overview`.

In [ ]:
%sql
SELECT stage_order, stage, users
FROM gold.funnel_overview
ORDER BY stage_order

## 2. Biggest drop-off: step-to-step conversion

`gold.step_conversion` gives the conversion rate between every consecutive
stage pair. The row with the **lowest `conversion_rate`** is the funnel's
single biggest drop-off point — usually the best place to focus a fix.

In [ ]:
%sql
SELECT step, from_users, to_users, conversion_rate
FROM gold.step_conversion
WHERE from_users IS NOT NULL
ORDER BY conversion_rate ASC

## 3. Channel volume vs. quality

Two views of the same acquisition channels, side by side: how many
bridge-linked downloads a channel drives (`gold.downloads_by_channel`) vs. how
many of those downloads actually convert into a paired hearing aid
(`gold.pairing_by_channel`). A channel can win on volume and still lose on
quality (or vice versa) — that gap is the interesting story here.

In [ ]:
%sql
SELECT acquisition_channel, linked_downloads
FROM gold.downloads_by_channel
ORDER BY linked_downloads DESC

In [ ]:
%sql
SELECT acquisition_channel, linked_downloads, paired_users, pairing_rate
FROM gold.pairing_by_channel
ORDER BY pairing_rate DESC

## 4. iOS vs. Android pairing, by market

`gold.pairing_by_platform_market` — the pairing rate of bridge-linked app
devices, cut by platform and market. Compare iOS vs. Android within the same
market (a platform gap that shows up in every market points at a product/
onboarding issue; a gap in one market only points at something local).

In [ ]:
%sql
SELECT market, platform, app_devices, paired_devices, pairing_rate
FROM gold.pairing_by_platform_market
ORDER BY market, platform

## 5. D30 retention by channel

`gold.d30_by_channel` — among bridge-linked, **non-censored** app devices,
the share still active 28-34 days after first open. Remember: this is
retention of the *linkable* population only — see the caveat at the top of
this notebook.

In [ ]:
%sql
SELECT acquisition_channel, eligible_users, retained_users, d30_retention_rate
FROM gold.d30_by_channel
ORDER BY d30_retention_rate DESC

## 6. Weekly test-start trend & linkable share by market

A quick look at seasonality/trend in test starts, and how much of the
funnel is measurable end-to-end per market (`gold.linkable_share_by_market`
— an approximation, see its `COMMENT ON` in `sql/medallion.sql`).

In [ ]:
%sql
SELECT week_start, test_starts
FROM gold.weekly_test_starts
ORDER BY week_start

In [ ]:
%sql
SELECT market, redirect_users, bridge_users, linkable_share
FROM gold.linkable_share_by_market
ORDER BY linkable_share DESC

## Summary

- The funnel's biggest drop-off is whichever step surfaces at the top of
  section 2 — re-run this notebook after any product change to see if it moved.
- Channel volume and channel quality (sections 3-4) do not always agree;
  size acquisition spend on `pairing_rate`, not `linked_downloads`, alone.
- Every cross-device number in this notebook is **linkable-population only**
  (consented + signed-in via `silver.v_attribution_eligible`) and D30 numbers
  additionally exclude right-censored devices — see `docs/knowledge/privacy.md`
  and `docs/knowledge/methodology.md` for the full reasoning.